<a href="https://colab.research.google.com/github/Usman-938/RAG_Chatbot/blob/main/Task4_RAG_Chatbot_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 4: Context-Aware Chatbot Using RAG (Retrieval-Augmented Generation)

**Objective:** Build a conversational chatbot that can remember context and retrieve external information during conversations.

**Student:** Muhammad Usman Ilyas  
**University:** Abdul Wali Khan University Mardan (AWKUM)  
**Program:** BS Artificial Intelligence  

---

## Problem Statement
Standard LLMs have no memory between turns and cannot access custom documents. This notebook implements a **RAG pipeline** that:
1. Embeds a custom knowledge corpus into a vector store (FAISS)
2. Retrieves relevant chunks for each user query
3. Maintains conversational history (context memory)
4. Generates grounded answers using an open-source LLM (via HuggingFace)

**No paid API key required — fully runnable on Google Colab (free tier).**

## Step 1 — Install Dependencies

In [ ]:
# Install all required packages with pinned compatible versions
!pip install -q -U langchain==0.3.25
!pip install -q -U langchain-community==0.3.23
!pip install -q -U langchain-huggingface==0.1.2
!pip install -q -U langchain-text-splitters==0.3.8
!pip install -q sentence-transformers faiss-cpu
!pip install -q transformers==4.51.3 accelerate
!pip install -q pypdf tiktoken
print('All packages installed successfully!')

## Step 2 — Import Libraries

In [ ]:
import os
import textwrap
import warnings
warnings.filterwarnings('ignore')

# Text splitter — standalone package in LangChain v0.2+
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector store and document loaders
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader

# Document schema moved to langchain_core in v0.2+
from langchain_core.documents import Document

# ConversationalRetrievalChain moved to langchain_community
from langchain.chains import ConversationalRetrievalChain

# Memory moved to langchain_community in v0.3+
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain.memory import ConversationBufferMemory

# Embeddings & LLM — HuggingFace (no API key needed)
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

# HuggingFace Transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import torch

print('All libraries imported successfully!')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device    : {DEVICE}')

## Step 3 — Dataset: Build Custom Knowledge Corpus

We create a multi-topic corpus covering **Artificial Intelligence**, **Machine Learning**, **Deep Learning**, and **Natural Language Processing**. This simulates a real-world knowledge base (Wikipedia pages / internal documents).

In [ ]:
# ---------------------------------------------------------------
# Custom knowledge corpus — simulates Wikipedia / internal docs
# ---------------------------------------------------------------
CORPUS = [
    {
        "title": "Artificial Intelligence Overview",
        "content": """
Artificial Intelligence (AI) is the simulation of human intelligence processes by machines,
especially computer systems. It encompasses learning (acquiring information and rules for using it),
reasoning (using rules to reach conclusions), and self-correction.

AI can be categorized as narrow AI (designed for a specific task) or general AI (human-level
intelligence across tasks). Current systems are all narrow AI. Applications include natural language
processing, computer vision, robotics, expert systems, and recommendation engines.

The Turing Test, proposed by Alan Turing in 1950, is a benchmark for machine intelligence. A machine
passes if a human evaluator cannot distinguish its responses from a human's. Modern AI has surpassed
humans on specific benchmarks like chess, Go, and image recognition.
"""
    },
    {
        "title": "Machine Learning",
        "content": """
Machine Learning (ML) is a subset of AI that gives systems the ability to learn and improve from
experience without being explicitly programmed. ML focuses on developing computer programs that can
access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: The model learns from labeled training data. Examples include linear
   regression, logistic regression, SVM, and decision trees.
2. Unsupervised Learning: The model finds patterns in unlabeled data. Examples include K-Means
   clustering, PCA, and autoencoders.
3. Reinforcement Learning: An agent learns by interacting with an environment and receiving rewards
   or penalties. Used in robotics and game playing (AlphaGo, DQN).

Key concepts: overfitting, underfitting, bias-variance tradeoff, cross-validation, regularization
(L1/L2), gradient descent, hyperparameter tuning, and feature engineering.
"""
    },
    {
        "title": "Deep Learning and Neural Networks",
        "content": """
Deep Learning is a subset of machine learning that uses artificial neural networks with multiple
layers (deep networks) to model complex patterns in data. It has revolutionized AI since 2012.

Key architectures:
- Feedforward Neural Networks (FNN): Simplest type; data flows in one direction.
- Convolutional Neural Networks (CNN): Excel at image tasks. Use convolutional layers to detect
  spatial features. Famous models: LeNet, AlexNet, VGG, ResNet, EfficientNet.
- Recurrent Neural Networks (RNN) / LSTM / GRU: Handle sequential data (time series, text).
  LSTMs solve the vanishing gradient problem with gating mechanisms.
- Transformers: Attention-based architecture (Vaswani et al., 2017). Foundation of modern NLP.
  Examples: BERT, GPT, T5, LLaMA.

Training involves forward pass, loss computation, backpropagation, and weight updates via optimizers
like SGD, Adam, and AdamW. Batch normalization and dropout are common regularization techniques.
"""
    },
    {
        "title": "Natural Language Processing",
        "content": """
Natural Language Processing (NLP) is a branch of AI that deals with the interaction between
computers and human language. It enables machines to read, understand, and generate text.

Core NLP tasks:
- Tokenization: Breaking text into words or subwords (BPE, WordPiece).
- Named Entity Recognition (NER): Identifying entities like names, dates, and locations.
- Sentiment Analysis: Determining emotional tone of text.
- Machine Translation: Translating between languages (e.g., Google Translate).
- Text Summarization: Extractive vs. abstractive summarization.
- Question Answering (QA): Finding answers in documents.

Word embeddings (Word2Vec, GloVe, FastText) represent words as dense vectors capturing semantic
relationships. BERT (Bidirectional Encoder Representations from Transformers) uses masked language
modeling for pre-training and has transformed NLP benchmarks. GPT models use autoregressive
language modeling and are the foundation of ChatGPT.
"""
    },
    {
        "title": "Retrieval-Augmented Generation (RAG)",
        "content": """
Retrieval-Augmented Generation (RAG) combines information retrieval with language generation.
Instead of relying solely on the LLM's parametric knowledge, RAG fetches relevant documents
from an external knowledge base and provides them as context for generation.

RAG Pipeline:
1. Indexing: Documents are split into chunks, embedded using an embedding model, and stored
   in a vector database (FAISS, Chroma, Pinecone, Weaviate).
2. Retrieval: At query time, the question is embedded and the top-k most similar chunks
   are retrieved using cosine similarity or approximate nearest neighbor (ANN) search.
3. Generation: The retrieved chunks + question are passed to an LLM as context to generate
   a grounded, factual answer.

Benefits of RAG: reduces hallucinations, enables access to private/updated knowledge,
no retraining required, and provides source attribution. RAG is the backbone of modern
enterprise AI assistants and document Q&A systems.
"""
    },
    {
        "title": "LangChain Framework",
        "content": """
LangChain is an open-source framework for building applications powered by large language models.
It provides abstractions for chaining LLM calls, managing memory, and integrating with external
tools and data sources.

Key LangChain components:
- Document Loaders: Load data from PDFs, CSVs, web pages, databases, etc.
- Text Splitters: Split documents into manageable chunks for embedding.
- Embeddings: Convert text to vector representations (OpenAI, HuggingFace, Cohere).
- Vector Stores: Index and retrieve embeddings (FAISS, Chroma, Pinecone).
- Chains: Compose multiple LLM calls and tools in sequence.
- Memory: Store and retrieve conversation history (Buffer, Summary, Token memory).
- Agents: LLMs that dynamically choose tools to use based on the task.

LangChain enables rapid development of RAG pipelines, chatbots, code assistants, data analysis
agents, and multi-modal applications.
"""
    },
    {
        "title": "Vector Databases and FAISS",
        "content": """
Vector databases store high-dimensional embeddings and enable fast similarity search.
They are essential for RAG, semantic search, recommendation systems, and anomaly detection.

FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and
clustering of dense vectors. It supports:
- Exact search (IndexFlatL2, IndexFlatIP) — brute-force, perfect recall
- Approximate search (IndexIVFFlat, IndexHNSW) — faster, slight recall loss
- GPU acceleration for large-scale search

Other popular vector databases: Chroma (lightweight, Python-native), Pinecone (managed cloud),
Weaviate (graph + vector hybrid), Qdrant, and Milvus.

Embedding models for text: sentence-transformers/all-MiniLM-L6-v2 is a popular lightweight
model (384 dimensions) with strong semantic similarity performance. Larger models like
text-embedding-ada-002 (OpenAI) and e5-large achieve higher accuracy at greater cost.
"""
    },
    {
        "title": "Conversational AI and Chatbot Design",
        "content": """
Conversational AI refers to technologies that enable machines to understand and respond to human
language in a dialogue format. Modern chatbots use LLMs fine-tuned on conversation data.

Context memory is critical for multi-turn conversations. Without memory, each user turn is treated
independently. Memory strategies:
- Buffer Memory: Stores all conversation turns (simple but grows unbounded).
- Summary Memory: Periodically summarizes older turns to compress context.
- Token Buffer Memory: Keeps the last N tokens of conversation.
- Entity Memory: Tracks specific named entities mentioned in the conversation.

Evaluation metrics for chatbots: BLEU (n-gram overlap), ROUGE (recall-oriented), BERTScore
(semantic similarity), Perplexity, and human evaluation (coherence, fluency, relevance).
RAG-specific metrics include context recall, context precision, answer faithfulness, and
answer relevance (measured by frameworks like RAGAS).
"""
    }
]

# Convert corpus to LangChain Document objects
documents = [
    Document(
        page_content=entry["content"].strip(),
        metadata={"source": entry["title"]}
    )
    for entry in CORPUS
]

print(f'Corpus loaded: {len(documents)} documents')
for i, doc in enumerate(documents):
    print(f'  [{i+1}] {doc.metadata["source"]} — {len(doc.page_content)} characters')

## Step 4 — Preprocessing: Text Splitting

In [ ]:
# Split documents into overlapping chunks for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,        # Max characters per chunk
    chunk_overlap=80,      # Overlap to preserve context at boundaries
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f'Text splitting complete!')
print(f'Original documents : {len(documents)}')
print(f'Total chunks       : {len(chunks)}')
print(f'Average chunk size : {sum(len(c.page_content) for c in chunks)//len(chunks)} characters')
print()
print('Sample chunk (first chunk):')
print('-' * 60)
print(chunks[0].page_content)
print(f'Source: {chunks[0].metadata["source"]}')

## Step 5 — Embedding Model & FAISS Vector Store

In [ ]:
# Load a lightweight but powerful sentence embedding model (no API key needed)
print('Loading embedding model...')
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'  # 384-dim, fast, strong semantic similarity

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': DEVICE},
    encode_kwargs={'normalize_embeddings': True}  # Cosine similarity
)

print(f'Embedding model loaded: {EMBEDDING_MODEL}')

# Build FAISS vector store from chunks
print('Building FAISS vector store...')
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f'Vector store built! Indexed {vectorstore.index.ntotal} vectors.')

# Save locally (optional — useful if you want to reload without re-embedding)
vectorstore.save_local('faiss_index')
print('Vector store saved to faiss_index/')

## Step 6 — Test Retriever

In [ ]:
# Configure retriever (top-3 most similar chunks per query)
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

# Test retrieval
test_query = 'What is retrieval augmented generation?'
retrieved_docs = retriever.get_relevant_documents(test_query)

print(f'Query: "{test_query}"')
print(f'Retrieved {len(retrieved_docs)} chunks:\n')
for i, doc in enumerate(retrieved_docs):
    print(f'--- Chunk {i+1} [Source: {doc.metadata["source"]}] ---')
    print(textwrap.fill(doc.page_content[:200], width=80))
    print()

## Step 7 — Load LLM (Google Flan-T5 via HuggingFace — Free, No API Key)

In [ ]:
LLM_MODEL = 'google/flan-t5-base'

print(f'Loading LLM: {LLM_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None
)
if DEVICE == 'cpu':
    model = model.to(DEVICE)

print('Creating HuggingFace text2text pipeline...')
hf_pipeline = pipeline(
    'text2text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True
)

# Wrap in LangChain-compatible LLM
llm = HuggingFacePipeline(pipeline=hf_pipeline)
print('LLM ready!')

## Step 8 — Build ConversationalRetrievalChain with Memory

In [ ]:
# Conversation memory — stores full chat history
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True,
    output_key='answer'
)

# Build the RAG chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False
)

print('RAG chain with memory built successfully!')
print('Components:')
print('  - Embedding model : sentence-transformers/all-MiniLM-L6-v2')
print('  - Vector store    : FAISS (in-memory + saved to disk)')
print('  - LLM             : google/flan-t5-base')
print('  - Memory          : ConversationBufferMemory')
print('  - Chain type      : ConversationalRetrievalChain')

## Step 9 — Chat Function

In [ ]:
def chat(question: str, show_sources: bool = True) -> str:
    """
    Send a question to the RAG chatbot and return the answer.

    Args:
        question: User's question string
        show_sources: Whether to display retrieved source chunks

    Returns:
        The chatbot's answer string
    """
    print(f'User: {question}')
    print('-' * 70)

    # Invoke the RAG chain (memory is updated automatically)
    result = rag_chain.invoke({'question': question})

    answer = result.get('answer', 'Sorry, I could not generate an answer.')
    source_docs = result.get('source_documents', [])

    # Display answer
    print(f'Bot: {textwrap.fill(answer, width=80)}')

    # Display sources if requested
    if show_sources and source_docs:
        print()
        print('Sources used:')
        seen = set()
        for doc in source_docs:
            src = doc.metadata.get('source', 'Unknown')
            if src not in seen:
                print(f'  • {src}')
                seen.add(src)

    print('=' * 70)
    return answer


def reset_memory():
    """Clear conversation history to start a new session."""
    memory.clear()
    print('Conversation memory cleared. Starting fresh session.')


print('Chat functions ready!')

## Step 10 — Evaluation: Multi-Turn Conversation Demo

We demonstrate the chatbot's **context awareness** through a 6-turn conversation where later questions reference earlier answers.

In [ ]:
# ---------------------------------------------------------------
# Multi-turn conversation demonstrating context memory + retrieval
# ---------------------------------------------------------------
print('=' * 70)
print('       RAG CHATBOT — MULTI-TURN CONVERSATION DEMO')
print('=' * 70)
print()

# Turn 1 — Broad question
chat('What is Artificial Intelligence?')

In [ ]:
# Turn 2 — Follow-up (tests context memory: 'it' refers to AI)
chat('What are the main types of machine learning within it?')

In [ ]:
# Turn 3 — Deep-dive into retrieval-specific topic
chat('How does Retrieval-Augmented Generation work?')

In [ ]:
# Turn 4 — Context-aware follow-up
chat('What vector databases are used in the pipeline you described?')

In [ ]:
# Turn 5 — NLP-related query
chat('Explain the difference between BERT and GPT models.')

In [ ]:
# Turn 6 — LangChain specifics
chat('What is LangChain and how does it help build chatbots like this one?')

## Step 11 — Inspect Conversation Memory

In [ ]:
# Display stored conversation history
print('Conversation History Stored in Memory:')
print('=' * 70)
chat_history = memory.load_memory_variables({})['chat_history']
print(f'Total turns in memory: {len(chat_history)}')
print()
for i, message in enumerate(chat_history):
    # HumanMessage / AIMessage classes from langchain_core
    role = 'Human' if 'Human' in type(message).__name__ else 'AI'
    content = message.content[:150] + '...' if len(message.content) > 150 else message.content
    print(f'[{i+1}] {role}: {content}')
    print()

## Step 12 — Evaluation Metrics

In [ ]:
import time
import numpy as np

# ---------------------------------------------------------------
# Automated evaluation on a Q&A test set
# ---------------------------------------------------------------
test_set = [
    {
        'question': 'What is a convolutional neural network used for?',
        'keywords': ['image', 'convolutional', 'spatial', 'CNN']
    },
    {
        'question': 'What is the difference between supervised and unsupervised learning?',
        'keywords': ['labeled', 'unlabeled', 'patterns', 'clustering']
    },
    {
        'question': 'What embedding model is commonly used for RAG systems?',
        'keywords': ['sentence', 'MiniLM', 'embedding', 'vector']
    },
    {
        'question': 'What is FAISS and what does it do?',
        'keywords': ['similarity', 'search', 'Facebook', 'vector']
    },
    {
        'question': 'How does memory work in a conversational chatbot?',
        'keywords': ['memory', 'history', 'context', 'conversation']
    }
]

# Reset memory for clean evaluation
reset_memory()

results = []
print('Running evaluation...\n')

for item in test_set:
    start = time.time()
    result = rag_chain.invoke({'question': item['question']})
    elapsed = time.time() - start

    answer = result.get('answer', '').lower()
    sources = result.get('source_documents', [])

    # Keyword hit rate as proxy for relevance
    keywords_found = sum(1 for kw in item['keywords'] if kw.lower() in answer)
    keyword_score = keywords_found / len(item['keywords'])

    results.append({
        'question': item['question'],
        'answer_length': len(result.get('answer', '')),
        'sources_retrieved': len(sources),
        'keyword_score': keyword_score,
        'response_time_s': round(elapsed, 2)
    })

# Print results table
print(f'{"Question":<55} {"KW Score":>10} {"Sources":>8} {"Time(s)":>8}')
print('-' * 85)
for r in results:
    q_short = r['question'][:52] + '...' if len(r['question']) > 52 else r['question']
    print(f'{q_short:<55} {r["keyword_score"]:>10.2f} {r["sources_retrieved"]:>8} {r["response_time_s"]:>8}')

print('-' * 85)
avg_kw = np.mean([r['keyword_score'] for r in results])
avg_time = np.mean([r['response_time_s'] for r in results])
avg_sources = np.mean([r['sources_retrieved'] for r in results])
print(f'{"AVERAGES":<55} {avg_kw:>10.2f} {avg_sources:>8.1f} {avg_time:>8.2f}')
print()
print(f'Average Keyword Relevance Score : {avg_kw:.2%}')
print(f'Average Response Time           : {avg_time:.2f} seconds')
print(f'Average Sources Retrieved       : {avg_sources:.1f} chunks')

## Step 13 — Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('RAG Chatbot — Evaluation Metrics', fontsize=14, fontweight='bold', y=1.02)

questions_short = [f'Q{i+1}' for i in range(len(results))]
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']

# Plot 1: Keyword Relevance Scores
bars = axes[0].bar(questions_short, [r['keyword_score'] for r in results],
                   color=colors, edgecolor='black', linewidth=0.8)
axes[0].axhline(avg_kw, color='red', linestyle='--', linewidth=1.5, label=f'Avg: {avg_kw:.2f}')
axes[0].set_title('Keyword Relevance Score', fontweight='bold')
axes[0].set_xlabel('Test Question')
axes[0].set_ylabel('Score (0–1)')
axes[0].set_ylim(0, 1.1)
axes[0].legend()
for bar, r in zip(bars, results):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{r["keyword_score"]:.2f}', ha='center', fontsize=9)

# Plot 2: Response Time
bars2 = axes[1].bar(questions_short, [r['response_time_s'] for r in results],
                    color=colors, edgecolor='black', linewidth=0.8)
axes[1].axhline(avg_time, color='red', linestyle='--', linewidth=1.5, label=f'Avg: {avg_time:.2f}s')
axes[1].set_title('Response Time per Question', fontweight='bold')
axes[1].set_xlabel('Test Question')
axes[1].set_ylabel('Time (seconds)')
axes[1].legend()
for bar, r in zip(bars2, results):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{r["response_time_s"]}s', ha='center', fontsize=9)

# Plot 3: Corpus document sizes
doc_names = [d.metadata['source'].split()[:2] for d in documents]
doc_labels = ['\n'.join(n) for n in doc_names]
doc_sizes = [len(d.page_content) for d in documents]
axes[2].barh(range(len(doc_sizes)), doc_sizes,
             color=plt.cm.tab10(np.linspace(0, 1, len(doc_sizes))),
             edgecolor='black', linewidth=0.8)
axes[2].set_yticks(range(len(doc_sizes)))
axes[2].set_yticklabels([d.metadata['source'][:25] for d in documents], fontsize=8)
axes[2].set_title('Corpus: Document Size (chars)', fontweight='bold')
axes[2].set_xlabel('Characters')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('rag_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to rag_evaluation.png')

In [ ]:
# RAG Pipeline Architecture Diagram
fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('RAG Pipeline Architecture', fontsize=14, fontweight='bold', pad=15)

# Define boxes: (x, y, w, h, label, color)
boxes = [
    (0.2,  1.5, 2.0, 2.0, 'Custom\nCorpus\n(Docs)', '#AED6F1'),
    (2.8,  1.5, 2.0, 2.0, 'Text\nSplitter\n(Chunks)', '#A9DFBF'),
    (5.4,  1.5, 2.0, 2.0, 'Embedding\nModel\n(MiniLM)', '#F9E79F'),
    (8.0,  1.5, 2.0, 2.0, 'FAISS\nVector\nStore', '#F1948A'),
    (10.6, 1.5, 2.0, 2.0, 'LLM\n(Flan-T5)\n+ Memory', '#D7BDE2'),
]
for (x, y, w, h, label, color) in boxes:
    rect = plt.Rectangle((x, y), w, h, color=color, ec='black', lw=1.5, zorder=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=9, fontweight='bold', zorder=3)

# Arrows
arrow_props = dict(arrowstyle='->', color='#2C3E50', lw=2)
for x_start, x_end in [(2.2, 2.8), (4.8, 5.4), (7.4, 8.0), (10.0, 10.6)]:
    ax.annotate('', xy=(x_end, 2.5), xytext=(x_start, 2.5),
                arrowprops=arrow_props, zorder=4)

# User query flow (bottom)
ax.annotate('User\nQuery', xy=(8.0, 1.5), xytext=(6.5, 0.3),
            arrowprops=dict(arrowstyle='->', color='blue', lw=1.5),
            fontsize=8, color='blue', ha='center')
ax.annotate('', xy=(12.6, 1.5), xytext=(12.6, 0.3),
            arrowprops=dict(arrowstyle='->', color='green', lw=1.5))
ax.text(12.6, 0.15, 'Answer', ha='center', fontsize=8, color='green', fontweight='bold')

plt.tight_layout()
plt.savefig('rag_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print('Architecture diagram saved to rag_architecture.png')

## Step 14 — Interactive Chat Loop (Run Manually)

In [ ]:
# ---------------------------------------------------------------
# Uncomment and run this cell to chat interactively in Colab
# Type 'quit' to exit, 'reset' to clear memory
# ---------------------------------------------------------------

# reset_memory()  # Fresh session

# print('RAG Chatbot ready! Type your question below.')
# print('Commands: "quit" to exit | "reset" to clear memory')
# print('=' * 70)

# while True:
#     user_input = input('You: ').strip()
#     if not user_input:
#         continue
#     if user_input.lower() == 'quit':
#         print('Goodbye!')
#         break
#     if user_input.lower() == 'reset':
#         reset_memory()
#         continue
#     chat(user_input)

print('Uncomment the block above to use the interactive chatbot.')

## Step 15 — Final Summary & Insights

In [ ]:
print('=' * 70)
print('       FINAL SUMMARY & INSIGHTS')
print('=' * 70)

summary = f"""
PROJECT: Context-Aware RAG Chatbot (Task 4)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ARCHITECTURE:
  • Corpus          : 8 custom AI/ML documents (custom knowledge base)
  • Chunks          : {len(chunks)} chunks (size=400, overlap=80)
  • Embedding Model : sentence-transformers/all-MiniLM-L6-v2 (384-dim)
  • Vector Store    : FAISS (IndexFlatL2 exact search)
  • LLM             : google/flan-t5-base (seq2seq, instruction-tuned)
  • Memory          : ConversationBufferMemory (full history)
  • Framework       : LangChain ConversationalRetrievalChain

EVALUATION RESULTS:
  • Avg Keyword Relevance Score : {avg_kw:.2%}
  • Avg Response Time           : {avg_time:.2f} seconds
  • Avg Source Chunks Retrieved : {avg_sources:.1f}
  • Multi-turn Context Memory   : ✓ Verified (6-turn demo)

KEY INSIGHTS:
  1. RAG significantly reduces hallucinations by grounding answers
     in retrieved documents rather than relying solely on LLM weights.
  2. Chunk overlap (80 chars) helps preserve context at chunk boundaries,
     improving answer coherence.
  3. ConversationBufferMemory enables multi-turn context: later questions
     can reference entities introduced in earlier turns.
  4. FAISS exact search ensures perfect recall at the cost of speed;
     approximate indexes (IVFFlat, HNSW) scale better for large corpora.
  5. Flan-T5 is a strong baseline for instruction-following without any
     API costs — ideal for academic and on-device deployment.

SKILLS GAINED:
  ✓ Conversational AI development (LangChain chains + memory)
  ✓ Document embedding and vector search (FAISS + sentence-transformers)
  ✓ Retrieval-Augmented Generation pipeline design
  ✓ LLM integration and deployment (HuggingFace Transformers)
"""
print(summary)